# 034: Doubly Linked List — Bidirectional Navigation, Sentinel Nodes, and Reference Cycle Management

## 1. THEORY LAYER

### Bidirectional Traversal and Sentinel Nodes
A Doubly Linked List (DLL) extends the singly linked list by adding a `prev` pointer to each node, allowing traversal in both directions (forward and backward).

To simplify boundary conditions (inserting or deleting at the ends), production systems often utilize **Sentinel Nodes** (dummy head and dummy tail):
- **Sentinel head**: Points to the first actual data node.
- **Sentinel tail**: Pointed to by the last actual data node.
- **Benefit**: Eliminates null checks when performing insertions and deletions, making operations faster and less error-prone.

### Memory Footprint
Each node in a DLL contains three references (data, next, prev) instead of two. This increases memory overhead on 64-bit systems by an additional 8 bytes per node, which is the cost of bidirectionality.

---

## 2. CPYTHON INTERNAL LAYER

### The Reference Cycle Threat
In a DLL, node A's `next` points to node B, and node B's `prev` points to node A. This bidirectional link creates a **mutual circular reference** between every adjacent pair of nodes.
- **Reference count limit**: The reference count of these nodes will never fall to 0, even if the list is discarded.
- **GC Dependency**: CPython's cyclic garbage collector is required to sweep and deallocate these nodes. This makes deletion slower than in singly linked lists. Alternatively, one pointer can be a `weakref` to break the cycle.

---


In [ ]:
import sys
import time
import weakref
import gc

print("=" * 70)
print("SECTION 1: Doubly Linked List Memory footprint")
print("=" * 70)

class DLLNodeMemory:
    def __init__(self, data):
        self.data = data
        self.next = None
        self.prev = None

n = DLLNodeMemory(42)
print(f"Base size of DLL Node on heap: {sys.getsizeof(n)} bytes")



---
## 3. COMPLETE A–Z METHOD COVERAGE

Below is the implementation of a Doubly Linked List using Sentinel Nodes:
- **Insertions**: Head, tail, and after an existing node.
- **Deletions**: Head, tail, and arbitrary node.
- **Reversal**: In-place swap of next and prev pointers.
- **CPython Reference management**: Explicitly clearing next/prev pointers on deletion to assist garbage collection.


In [ ]:
class DLLNode:
    """A single node in a Doubly Linked List."""
    def __init__(self, data):
        self.data = data
        self.next = None
        self.prev = None

    def __repr__(self):
        return f"DLLNode({self.data})"

class DoublyLinkedList:
    """CPython-style Doubly Linked List Engine with Sentinel Head/Tail."""
    def __init__(self):
        # Initialize sentinel (dummy) nodes
        self.head = DLLNode(None)
        self.tail = DLLNode(None)
        self.head.next = self.tail
        self.tail.prev = self.head
        self.size = 0

    def insert_at_head(self, data):
        """Inserts a node right after the sentinel head. Time: O(1)."""
        new_node = DLLNode(data)
        first_node = self.head.next
        
        # Link new node between head and first_node
        new_node.next = first_node
        new_node.prev = self.head
        
        self.head.next = new_node
        first_node.prev = new_node
        
        self.size += 1

    def insert_at_tail(self, data):
        """Inserts a node right before the sentinel tail. Time: O(1)."""
        new_node = DLLNode(data)
        last_node = self.tail.prev
        
        # Link new node between last_node and tail
        new_node.next = self.tail
        new_node.prev = last_node
        
        last_node.next = new_node
        self.tail.prev = new_node
        
        self.size += 1

    def delete_node(self, node):
        """Removes a specific node from the list. Time: O(1)."""
        if node is self.head or node is self.tail:
            return  # Cannot delete sentinels
            
        before = node.prev
        after = node.next
        
        # Link neighbors directly
        before.next = after
        after.prev = before
        
        # Explicitly clear links to prevent cyclic memory leaks
        node.next = None
        node.prev = None
        
        self.size -= 1
        return node.data

    def delete_head(self):
        """Removes and returns the first actual data node. Time: O(1)."""
        if self.size == 0:
            raise IndexError("Delete from empty list")
        return self.delete_node(self.head.next)

    def delete_tail(self):
        """Removes and returns the last actual data node. Time: O(1)."""
        if self.size == 0:
            raise IndexError("Delete from empty list")
        return self.delete_node(self.tail.prev)

    def reverse(self):
        """Reverses the list in-place by swapping next and prev pointers. Time: O(1)."""
        current = self.head
        while current is not None:
            # Swap next and prev pointers
            temp = current.next
            current.next = current.prev
            current.prev = temp
            # Move backward (which was next before swap)
            current = temp
            
        # Swap sentinel head and tail pointers
        temp = self.head
        self.head = self.tail
        self.tail = temp
        
    def __len__(self):
        return self.size

    def __repr__(self):
        nodes = []
        current = self.head.next
        while current is not self.tail:
            nodes.append(str(current.data))
            current = current.next
        return "None <-> " + " <-> ".join(nodes) + " <-> None"



In [ ]:
# Verify Level 1 execution
print("\n=== Doubly Linked List Verification ===")
dll = DoublyLinkedList()
dll.insert_at_head(20)
dll.insert_at_head(10)
dll.insert_at_tail(30)

print(f"Created List:  {dll}")
dll.reverse()
print(f"Reversed List: {dll}")



---
## 4. IMPLEMENTATION LAYER

### Level 2: Performance Benchmark vs CPython Lists
We compare inserting elements at the tail of a DLL vs a standard list:
- **CPython List**: $O(1)$ amortized (resizing cost is amortized).
- **Doubly Linked List**: $O(1)$ absolute (since sentinel tail access allows instant insert).


In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Performance Benchmarks")
print("=" * 70)

N = 25000

# 1. Native List Benchmark
t0 = time.perf_counter()
native_list = []
for i in range(N):
    native_list.append(i)
t_native = (time.perf_counter() - t0) * 1000

# 2. DLL Benchmark
t0 = time.perf_counter()
dll_list = DoublyLinkedList()
for i in range(N):
    dll_list.insert_at_tail(i)
t_dll = (time.perf_counter() - t0) * 1000

print(f"Native list append:  {t_native:.2f} ms")
print(f"Doubly linked list append:  {t_dll:.2f} ms")



---
### Level 3: Advanced Usage Systems — Production LRU Cache

A production-grade LRU Cache uses a **Doubly Linked List** because it allows $O(1)$ removal of *any* arbitrary node once its reference is looked up in the Hash Map (in a Singly Linked List, removal is $O(n)$ because you must search for the previous node).


In [ ]:
class ProductionLRUCache:
    """LRU Cache powered by a Doubly Linked List and a Hash Map. Time: O(1) for all ops."""
    def __init__(self, capacity):
        self.capacity = capacity
        # Map stores key -> DLLNode references
        self.map = {}
        self.list = DoublyLinkedList()

    def get(self, key):
        if key not in self.map:
            return None
        node = self.map[key]
        # Move node to head (most recently used)
        self.list.delete_node(node)
        self.list.insert_at_head(node.data)
        # Map now references the new node at head
        self.map[key] = self.list.head.next
        return node.data[1]

    def put(self, key, value):
        if key in self.map:
            # Remove old node
            self.list.delete_node(self.map[key])
            
        elif len(self.map) >= self.capacity:
            # Evict least recently used (node right before sentinel tail)
            lru_node = self.list.tail.prev
            lru_key = lru_node.data[0]
            self.list.delete_node(lru_node)
            del self.map[lru_key]
            print(f"Evicted key: {lru_key}")

        # Insert at head
        self.list.insert_at_head((key, value))
        self.map[key] = self.list.head.next

print("\n=== Level 3: Production LRU Cache ===")
cache = ProductionLRUCache(capacity=3)
cache.put("user1", "Alice")
cache.put("user2", "Bob")
cache.put("user3", "Charlie")
cache.get("user1")  # user1 accessed
cache.put("user4", "Dave")  # Evicts user2

print(f"Cache state (newest first): {cache.list}")



---
## 5. EXPERIMENTATION LAYER
### The Reference Cycle Leak Experiment
We demonstrate how bidirectional links prevent immediate deallocation by tracing CPython reference counts.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Reference Cycle Analysis")
print("=" * 70)

class TestNode:
    def __init__(self, data):
        self.data = data
        self.next = None
        self.prev = None
    def __del__(self):
        pass  # Tracker

node_a = TestNode(1)
node_b = TestNode(2)

node_a.next = node_b
node_b.prev = node_a

# Get reference count (includes temporary ref from getrefcount)
print(f"node_a ref count: {sys.getrefcount(node_a)}") # High due to cross link
del node_a
del node_b
print("References deleted. Memory is still occupied until gc.collect() sweeps the cycle.")



---
## 6. VISUALIZATION LAYER
```
Doubly Linked List Structure:
       +------ next ------->     +------ next ------->
Head <->  Node (10)  <------->   Node (20)  <-------> Tail
       <------ prev -------+     <------ prev -------+
```
---
## 7. INTERVIEW MASTER LAYER

### 100 Scenario-Based Linked List Interview Q&As (Summary Selection)

1. **Why is a Doubly Linked List preferred over a Singly Linked List for LRU Cache?**
   - Because a DLL allows deleting any node in $O(1)$ time given its reference, as you can access `node.prev` and `node.next` instantly. In a singly linked list, you have to traverse from head to find the node before target node, making deletion $O(n)$.
2. **What are sentinel nodes and why are they used?**
   - Sentinel nodes are dummy nodes at the head and tail that contain no data. They simplify insertion and deletion code by removing null-check exceptions.
3. **How does C3 Linearization resolve multiple inheritance conflicts?**
   - It merges parent class lists of classes while preserving order and monotonicity.
4. **How do you reverse a Doubly Linked List?**
   - Iterate through the list and swap the `next` and `prev` pointers for every node, then swap the head and tail pointer references.
5. **Explain the memory overhead difference between DLL and SLL.**
   - DLL nodes store an additional pointer reference (`prev`), costing an extra 8 bytes per node on 64-bit platforms.
